# 06 - HGD: EmerG x DGD Hybrid

Train and evaluate HGD (Hybrid Graph Diffusion) on the shared MovieLens-1M
cold-start protocol. HGD keeps EmerG's bounded dense adjacency, forced
self-loops and additive residual propagation, and adds DGD's learnable
graph-power mixing, row-wise sparsemax and multiplicative field gating as gates
that open only if validation rewards them. The design target is to beat the
notebook-04 EmerG baseline on the frozen evaluation protocol.


## Notebook Linkage and Work Plan

**Input from notebook 02:** the `ml1m-coldstart-v1` protocol manifest plus
verified `tuning_train`, `final_train`, `validation_tasks`, `evaluation_tasks`,
`users`, and `items` artifacts. HGD reads the protocol directly, exactly as
notebooks 03, 04 and 05 do.

**Optional input from notebook 04:** the `emerg-v1` generations. The final cell
pairs them with this notebook's bundles by seed to read the delta at run time.
It is a sanity read; notebook 08 remains the arbiter for published comparisons.

**What this notebook does:**
1. Verify and load the notebook-02 protocol bundle.
2. Fit the same leakage-safe side-feature encoders used by notebooks 04 and 05,
   so the feature vocabularies hash identically and the comparison isolates the
   method.
3. Implement HGD: item-specific graph generation, learnable graph-power mixing,
   dense/sparse graph fusion with forced self-loops, additive plus gated
   multiplicative propagation, and a side-information cold-start item prior.
4. Run unit and gradient checks, including the reduction check that HGD with all
   gates closed reproduces EmerG's forward pass exactly.
5. For each target seed, tune on validation only, refit on `final_train`,
   evaluate Cold/Warm A/B/C with frozen thresholds, and publish a separate
   immutable result bundle.
6. Compare against the notebook-04 EmerG bundles on the same protocol.


## Method: what is taken from where, and why

Notebook 05 replaced EmerG's graph construction wholesale, and on the frozen
protocol it did not clear the baseline: its published Cold F1 is the weakest of
the four phases, which is where the sigmoid-plus-self-loop adjacency of EmerG is
doing real work. HGD therefore treats EmerG as the floor rather than the rival.

| Component | EmerG (04) | DGD (05) | HGD (06) |
| --- | --- | --- | --- |
| Adjacency | `sigmoid` of generated logits, bounded, magnitudes kept | row `sparsemax`, sparse but row-stochastic, magnitudes discarded | convex fusion `(1 - a) * dense + a * sparse`, `a = sigmoid(fusion_logits)` learned per layer |
| Self-loops | forced to 1 on the diagonal | none, sparsemax may zero a field's own edge | forced to 1 after fusion, so sparsity can never delete a field's own contribution |
| Receptive field | first-order graph reused at every layer | learnable simplex mixture of powers 1..l | learnable mixture, initialised with its mass on power 1 (EmerG's choice) |
| Propagation | `h <- h + relu(A W h)`, depth accumulates | `h <- h * (1 + tanh(A V h0))`, no depth accumulation | both: additive residual on `h`, then `h <- h * (1 + g * tanh(A V h0))` with `g` a small per-layer gate |
| Cold item id | untrained random vector at eval | untrained random vector at eval | zero-initialised id summed with an embedding synthesised from audited side info, trained under id dropout |

**Why this should beat EmerG rather than merely differ from it.** Set the fusion
gates `a -> 0`, the modulation gates `g -> 0`, the power mixture to power 1, the
dense temperature to `sqrt(num_fields)` and the synthesis head to zero: the
forward pass is then EmerG's, parameter for parameter. `emerg_reduction_error`
asserts this to `< 1e-4` before any training happens. Every addition is
initialised near that point (`a = sigmoid(-2) ~ 0.12`, `g = 0.1`, ~98% of the
power mass on the first-order graph), so training starts inside EmerG's regime
and can only spend gate budget where the loss pays for it. The remaining risk is
optimisation, not expressiveness, and validation-based config selection is the
control for it.

**Why the cold-start prior is the highest-leverage addition.** `final_train`
excludes every evaluation item, so at Cold the id row of an evaluation item has
received no gradient. EmerG and DGD both score it with its random initialisation
- noise in the field that the graph generator also reads. HGD zero-initialises
the id table and always sums an embedding synthesised from `release_year`,
`genres` and `title`, and applies id dropout during training. An unseen item is
then exactly the regime the model trained under, and the Warm phases still adapt
the local id residual and graph delta on the same budget as the baselines.

**Fairness.** Feature encoders, protocol bundle, phase order, epochs, sampled
rows per epoch, batch size, learning rates, weight decay, warm steps, threshold
policy and export contract are unchanged from notebooks 04 and 05. HGD adds
parameters and per-batch compute; it does not add data, epochs or warm-up steps.


In [ ]:
from __future__ import annotations

import copy
import hashlib
import importlib.util
import json
import os
import platform
import random
import re
import shutil
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

REQUIRED_PACKAGES = ["numpy", "pandas", "torch", "IPython"]
MISSING_PACKAGES = [
    package for package in REQUIRED_PACKAGES if importlib.util.find_spec(package) is None
]
if MISSING_PACKAGES:
    raise RuntimeError(
        "Notebook 06 requires these packages in the active kernel: "
        + ", ".join(MISSING_PACKAGES)
        + ". Run it in the project ML/Kaggle environment used for model notebooks."
    )

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import Markdown, display
from torch import nn


def show_records(records: Iterable[dict[str, Any]]) -> None:
    display(pd.DataFrame(list(records)))


def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_inside(root: Path, relative_path: str) -> Path:
    resolved_root = root.resolve()
    resolved = (resolved_root / relative_path).resolve()
    resolved.relative_to(resolved_root)
    return resolved


def project_root(start: Path) -> Path:
    override = os.environ.get("COLDSTART_PROJECT_ROOT")
    if override:
        return Path(override).expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate.resolve()
    return start.resolve()


EXECUTION_CONTEXT = "kaggle" if Path("/kaggle/input").exists() else "local"
PROJECT_ROOT = project_root(Path.cwd())
WORKSPACE_ROOT = Path(
    os.environ.get(
        "COLDSTART_WORKSPACE_ROOT",
        "/kaggle/working" if EXECUTION_CONTEXT == "kaggle" else PROJECT_ROOT / ".notebook",
    )
).expanduser().resolve()
INPUT_ROOT = Path(
    os.environ.get(
        "COLDSTART_INPUT_ROOT",
        "/kaggle/input" if EXECUTION_CONTEXT == "kaggle" else PROJECT_ROOT / "data",
    )
).expanduser().resolve()
ARTIFACT_ROOT = Path(
    os.environ.get("COLDSTART_ARTIFACT_ROOT", WORKSPACE_ROOT / "artifacts")
).expanduser().resolve()
PROTOCOL_RELATIVE_MANIFEST = Path("protocols/ml-1m/coldstart-v1/manifest.json")
MODEL_OUTPUT_ROOT = ARTIFACT_ROOT / "models" / "ml-1m" / "hgd-v1"
FAST_DEV_RUN = os.environ.get("COLDSTART_FAST_DEV_RUN", "0") == "1"

requested_device = os.environ.get("COLDSTART_DEVICE")
if requested_device:
    DEVICE = torch.device(requested_device)
    if DEVICE.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("COLDSTART_DEVICE requests CUDA, but torch.cuda is unavailable")
else:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TARGET_SEEDS = [2025, 7788, 9999, 3407, 4517]
PHASE_ORDER = ["Cold", "Warm A", "Warm B", "Warm C"]
FIELD_ORDER = [
    "user_id", "gender", "age", "occupation", "zip_code",
    "item_id", "release_year", "genres", "title",
]
ITEM_GRAPH_FIELDS = ["item_id", "release_year", "genres", "title"]
ITEM_SIDE_FIELDS = ["release_year", "genres", "title"]
MAX_TITLE_TOKENS = 8
MAX_GENRE_TOKENS = 6

base_epochs = int(os.environ.get("COLDSTART_HGD_EPOCHS", "4"))
base_sample_rows = int(os.environ.get("COLDSTART_HGD_SAMPLE_ROWS", "262144"))
base_warm_steps = int(os.environ.get("COLDSTART_HGD_WARM_STEPS", "3"))
if FAST_DEV_RUN:
    base_epochs = min(base_epochs, 1)
    base_sample_rows = min(base_sample_rows, 32768)
    base_warm_steps = min(base_warm_steps, 1)

def make_run_config(seed: int) -> dict[str, Any]:
    return {
        "schema_version": "hgd-model-v1",
        "seed": int(seed),
        "fast_dev_run": FAST_DEV_RUN,
        "phase_order": list(PHASE_ORDER),
        "field_order": list(FIELD_ORDER),
        "item_graph_fields": list(ITEM_GRAPH_FIELDS),
        "item_side_fields": list(ITEM_SIDE_FIELDS),
        "max_title_tokens": MAX_TITLE_TOKENS,
        "max_genre_tokens": MAX_GENRE_TOKENS,
        "graph": (
            "per-layer convex fusion of an EmerG dense sigmoid adjacency with a DGD "
            "row-sparsemax adjacency, both read off the same learnable mixture of graph "
            "powers 1..l, with EmerG self-loops forced after fusion"
        ),
        "propagation": (
            "EmerG additive residual message passing over the fused graph plus a gated "
            "DGD multiplicative modulation computed from the layer-0 field embeddings"
        ),
        "cold_start_prior": (
            "item id embedding zero-initialised and always summed with an embedding "
            "synthesised from audited item side information; id dropout during training"
        ),
        "budget_parity": (
            "epochs, epoch_sample_rows, batch_size, learning rates, weight decay and "
            "warm_steps match notebooks 04 and 05 so the comparison isolates the method"
        ),
        "candidate_configs": [
            {
                "embedding_dim": 16,
                "hidden_dim": 64,
                "gnn_layers": 2,
                "learning_rate": 0.001,
                "warm_learning_rate": 0.01,
                "weight_decay": 1e-6,
                "sparsemax_scale": 5.0,
                "fusion_logit_init": -2.0,
                "modulation_gate_init": 0.1,
                "power_prior_bias": 4.0,
                "id_dropout": 0.3,
                "epochs": base_epochs,
                "epoch_sample_rows": base_sample_rows,
                "batch_size": 4096,
                "warm_steps": base_warm_steps,
            },
            {
                "embedding_dim": 32,
                "hidden_dim": 96,
                "gnn_layers": 3,
                "learning_rate": 0.001,
                "warm_learning_rate": 0.01,
                "weight_decay": 1e-6,
                "sparsemax_scale": 5.0,
                "fusion_logit_init": -2.0,
                "modulation_gate_init": 0.1,
                "power_prior_bias": 4.0,
                "id_dropout": 0.3,
                "epochs": base_epochs,
                "epoch_sample_rows": base_sample_rows,
                "batch_size": 4096,
                "warm_steps": base_warm_steps,
            },
        ][:1 if FAST_DEV_RUN else 2],
        "feature_policy": "fit categorical/text vocabularies on tuning-visible users/items; use PAD/UNK",
        "warmup_policy": "adapt local item embedding and graph delta sequentially: A, then B only, then C only",
    }


def protocol_candidates() -> list[tuple[Path, Path]]:
    candidates: list[tuple[Path, Path]] = []
    explicit = os.environ.get("COLDSTART_PROTOCOL_ROOT")
    roots = [Path(explicit).expanduser()] if explicit else []
    roots.extend([PROJECT_ROOT / ".notebook" / "artifacts", ARTIFACT_ROOT])

    for root in roots:
        pointer = root / PROTOCOL_RELATIVE_MANIFEST
        if pointer.is_file():
            candidates.append((root.resolve(), pointer.resolve()))
        direct = root / "manifest.json"
        if root.name == "coldstart-v1" and direct.is_file():
            candidates.append((root.parents[2].resolve(), direct.resolve()))

    if INPUT_ROOT.is_dir():
        for pointer in sorted(INPUT_ROOT.rglob("manifest.json")):
            parent = pointer.parent
            if (
                parent.name == "coldstart-v1"
                and parent.parent.name == "ml-1m"
                and parent.parent.parent.name == "protocols"
            ):
                candidates.append((pointer.parents[3].resolve(), pointer.resolve()))

    unique: list[tuple[Path, Path]] = []
    seen: set[str] = set()
    for root, pointer in candidates:
        key = str(pointer)
        if key not in seen:
            seen.add(key)
            unique.append((root, pointer))
    return unique


def load_verified_protocol(root: Path, pointer: Path) -> tuple[dict[str, Any], dict[str, pd.DataFrame]]:
    pointer_bytes = pointer.read_bytes()
    manifest = json.loads(pointer_bytes)
    if manifest.get("protocol_schema_version") != "ml1m-coldstart-v1":
        raise ValueError(f"Unexpected protocol schema: {manifest.get('protocol_schema_version')!r}")
    if manifest.get("protocol_status") != "PASS":
        raise ValueError(f"Protocol status is not PASS: {manifest.get('protocol_status')!r}")
    checks = manifest.get("checks")
    if (
        not isinstance(checks, list)
        or not checks
        or not all(isinstance(row, dict) and row.get("status") == "PASS" for row in checks)
    ):
        raise ValueError("One or more notebook-02 protocol checks did not pass")

    bundle_id = manifest.get("bundle_id")
    if (
        not isinstance(bundle_id, str)
        or not bundle_id
        or bundle_id in {".", ".."}
        or Path(bundle_id).name != bundle_id
    ):
        raise ValueError(f"Invalid protocol bundle id: {bundle_id!r}")
    bundle_manifest = resolve_inside(root, manifest["bundle_manifest"])
    expected_bundle_manifest = (
        pointer.parent / "generations" / bundle_id / "manifest.json"
    ).resolve()
    if bundle_manifest != expected_bundle_manifest:
        raise ValueError("Protocol bundle_manifest is not the immutable generation manifest")
    if bundle_manifest.read_bytes() != pointer_bytes:
        raise ValueError("Protocol pointer and immutable generation manifest differ")

    tables: dict[str, pd.DataFrame] = {}
    for name, artifact in manifest["artifacts"].items():
        path = resolve_inside(root, artifact["path"])
        if not path.is_file() or sha256_file(path) != artifact["sha256"]:
            raise ValueError(f"Protocol artifact verification failed: {name}")
        schema = manifest["output_schemas"][name]
        table = pd.read_csv(path, dtype=schema["read_csv_dtypes"])
        if list(table.columns) != schema["columns"] or len(table) != artifact["rows"]:
            raise ValueError(f"Protocol table contract failed: {name}")
        tables[name] = table
    return manifest, tables


PROTOCOL_ERRORS: list[str] = []
PROTOCOL_ROOT = None
PROTOCOL_POINTER = None
PROTOCOL_MANIFEST = None
TABLES = None
for candidate_root, candidate_pointer in protocol_candidates():
    try:
        PROTOCOL_MANIFEST, TABLES = load_verified_protocol(candidate_root, candidate_pointer)
        PROTOCOL_ROOT, PROTOCOL_POINTER = candidate_root, candidate_pointer
        break
    except Exception as error:
        PROTOCOL_ERRORS.append(f"{candidate_pointer}: {error}")

if PROTOCOL_MANIFEST is None or TABLES is None or PROTOCOL_POINTER is None:
    raise RuntimeError(
        "No valid notebook-02 protocol bundle found. Set COLDSTART_PROTOCOL_ROOT. "
        + " | ".join(PROTOCOL_ERRORS)
    )

TUNING_TRAIN = TABLES["tuning_train"]
FINAL_TRAIN = TABLES["final_train"]
VALIDATION_TASKS = TABLES["validation_tasks"]
EVALUATION_TASKS = TABLES["evaluation_tasks"]
USERS = TABLES["users"].sort_values("user_idx").reset_index(drop=True)
ITEMS = TABLES["items"].sort_values("item_idx").reset_index(drop=True)
N_USERS = int(USERS["user_idx"].max()) + 1
N_ITEMS = int(ITEMS["item_idx"].max()) + 1
PHASES = tuple(PHASE_ORDER)
PHASE_INCREMENT_ROLE = {"Warm A": "warm_a", "Warm B": "warm_b", "Warm C": "warm_c"}
PROTOCOL_POINTER_SHA256 = sha256_file(PROTOCOL_POINTER)

show_records(
    [
        {
            "execution_context": EXECUTION_CONTEXT,
            "python": platform.python_version(),
            "torch": torch.__version__,
            "device": str(DEVICE),
            "protocol_bundle": PROTOCOL_MANIFEST["bundle_id"],
            "protocol_pointer_sha256": PROTOCOL_POINTER_SHA256,
            "users": N_USERS,
            "items": N_ITEMS,
            "validation_query_rows": int(VALIDATION_TASKS["role"].eq("query").sum()),
            "evaluation_query_rows": int(EVALUATION_TASKS["role"].eq("query").sum()),
            "target_seeds": TARGET_SEEDS,
        }
    ]
)

In [ ]:
TOKEN_RE = re.compile(r"[a-z0-9]+")
PAD = "<PAD>"
UNK = "<UNK>"


def make_vocab(values: Iterable[Any]) -> dict[str, int]:
    vocab = {PAD: 0, UNK: 1}
    for value in sorted({str(item) for item in values if pd.notna(item)}):
        if value and value not in vocab:
            vocab[value] = len(vocab)
    return vocab


def encode_scalar(vocab: dict[str, int], value: Any) -> int:
    return vocab.get(str(value), vocab[UNK]) if pd.notna(value) else vocab[UNK]


def title_tokens(title: Any) -> list[str]:
    return TOKEN_RE.findall(str(title).lower())


def genre_tokens(genres: Any) -> list[str]:
    return [token for token in str(genres).split("|") if token]


def encode_sequence(vocab: dict[str, int], tokens: list[str], max_len: int) -> list[int]:
    encoded = [vocab.get(token, vocab[UNK]) for token in tokens[:max_len]]
    return encoded + [vocab[PAD]] * (max_len - len(encoded))


class FeatureStore:
    def __init__(self, users: pd.DataFrame, items: pd.DataFrame, train: pd.DataFrame):
        train_user_ids = set(train["user_idx"].astype(int))
        train_item_ids = set(train["item_idx"].astype(int))
        train_users = users[users["user_idx"].isin(train_user_ids)]
        train_items = items[items["item_idx"].isin(train_item_ids)]

        self.gender_vocab = make_vocab(train_users["gender"])
        self.age_vocab = make_vocab(train_users["age"])
        self.occupation_vocab = make_vocab(train_users["occupation"])
        self.zip_vocab = make_vocab(train_users["zip_code"])
        self.genre_vocab = make_vocab(
            token for value in train_items["genres"] for token in genre_tokens(value)
        )
        self.title_vocab = make_vocab(
            token for value in train_items["title"] for token in title_tokens(value)
        )

        train_years = pd.to_numeric(train_items["release_year"], errors="coerce").astype("float32")
        self.release_mean = float(train_years.mean())
        self.release_std = float(train_years.std() if train_years.std() > 0 else 1.0)

        users = users.sort_values("user_idx").reset_index(drop=True)
        items = items.sort_values("item_idx").reset_index(drop=True)
        self.user_gender = np.array([encode_scalar(self.gender_vocab, x) for x in users["gender"]], dtype=np.int64)
        self.user_age = np.array([encode_scalar(self.age_vocab, x) for x in users["age"]], dtype=np.int64)
        self.user_occupation = np.array([encode_scalar(self.occupation_vocab, x) for x in users["occupation"]], dtype=np.int64)
        self.user_zip = np.array([encode_scalar(self.zip_vocab, x) for x in users["zip_code"]], dtype=np.int64)

        years = pd.to_numeric(items["release_year"], errors="coerce").astype("float32")
        years = years.fillna(self.release_mean)
        self.item_release = ((years.to_numpy(dtype=np.float32) - self.release_mean) / self.release_std).astype(np.float32)
        self.item_genres = np.array(
            [
                encode_sequence(
                    self.genre_vocab, genre_tokens(value), MAX_GENRE_TOKENS
                )
                for value in items["genres"]
            ],
            dtype=np.int64,
        )
        self.item_titles = np.array(
            [
                encode_sequence(
                    self.title_vocab, title_tokens(value), MAX_TITLE_TOKENS
                )
                for value in items["title"]
            ],
            dtype=np.int64,
        )
        self.field_sizes = {
            "user_id": N_USERS,
            "gender": len(self.gender_vocab),
            "age": len(self.age_vocab),
            "occupation": len(self.occupation_vocab),
            "zip_code": len(self.zip_vocab),
            "item_id": N_ITEMS,
            "genres": len(self.genre_vocab),
            "title": len(self.title_vocab),
        }

    def arrays(self, table: pd.DataFrame) -> dict[str, np.ndarray]:
        return {
            "source_row": table["source_row"].to_numpy(dtype=np.int64),
            "user_id": table["user_id"].to_numpy(dtype=np.int64),
            "user_idx": table["user_idx"].to_numpy(dtype=np.int64),
            "item_id": table["item_id"].to_numpy(dtype=np.int64),
            "item_idx": table["item_idx"].to_numpy(dtype=np.int64),
            "label": table["label"].to_numpy(dtype=np.float32),
        }

    def batch(self, arrays: dict[str, np.ndarray], rows: np.ndarray) -> tuple[dict[str, torch.Tensor], torch.Tensor]:
        user_idx = arrays["user_idx"][rows]
        item_idx = arrays["item_idx"][rows]
        features = {
            "user_id": torch.as_tensor(user_idx, dtype=torch.long, device=DEVICE),
            "gender": torch.as_tensor(self.user_gender[user_idx], dtype=torch.long, device=DEVICE),
            "age": torch.as_tensor(self.user_age[user_idx], dtype=torch.long, device=DEVICE),
            "occupation": torch.as_tensor(self.user_occupation[user_idx], dtype=torch.long, device=DEVICE),
            "zip_code": torch.as_tensor(self.user_zip[user_idx], dtype=torch.long, device=DEVICE),
            "item_id": torch.as_tensor(item_idx, dtype=torch.long, device=DEVICE),
            "release_year": torch.as_tensor(self.item_release[item_idx], dtype=torch.float32, device=DEVICE),
            "genres": torch.as_tensor(self.item_genres[item_idx], dtype=torch.long, device=DEVICE),
            "title": torch.as_tensor(self.item_titles[item_idx], dtype=torch.long, device=DEVICE),
        }
        labels = torch.as_tensor(arrays["label"][rows], dtype=torch.float32, device=DEVICE)
        return features, labels

    def contract(self) -> dict[str, Any]:
        vocabs = {
            "gender": self.gender_vocab,
            "age": self.age_vocab,
            "occupation": self.occupation_vocab,
            "zip_code": self.zip_vocab,
            "genres": self.genre_vocab,
            "title": self.title_vocab,
        }
        return {
            "schema_version": "hgd-feature-contract-v1",
            "field_order": list(FIELD_ORDER),
            "item_graph_fields": list(ITEM_GRAPH_FIELDS),
            "max_title_tokens": MAX_TITLE_TOKENS,
            "max_genre_tokens": MAX_GENRE_TOKENS,
            "release_year_mean": self.release_mean,
            "release_year_std": self.release_std,
            "field_sizes": self.field_sizes,
            "vocab_sha256": {
                name: sha256_bytes(
                    json.dumps(vocab, sort_keys=True, allow_nan=False).encode()
                )
                for name, vocab in vocabs.items()
            },
            "vocab_sizes": {name: len(vocab) for name, vocab in vocabs.items()},
            "vocabs": vocabs,
        }


FEATURES = FeatureStore(USERS, ITEMS, TUNING_TRAIN)
FEATURE_CONTRACT = FEATURES.contract()
display(pd.DataFrame([FEATURE_CONTRACT["field_sizes"]]))
show_records(
    [
        {
            "feature_contract": FEATURE_CONTRACT["schema_version"],
            "title_vocab": FEATURE_CONTRACT["vocab_sizes"]["title"],
            "genre_vocab": FEATURE_CONTRACT["vocab_sizes"]["genres"],
            "zip_vocab": FEATURE_CONTRACT["vocab_sizes"]["zip_code"],
            "release_year_mean": round(FEATURE_CONTRACT["release_year_mean"], 3),
            "release_year_std": round(FEATURE_CONTRACT["release_year_std"], 3),
        }
    ]
)

In [ ]:
ITEM_GRAPH_POSITIONS = [FIELD_ORDER.index(name) for name in ITEM_GRAPH_FIELDS]
ITEM_SIDE_POSITIONS = [FIELD_ORDER.index(name) for name in ITEM_SIDE_FIELDS]


def sparsemax(logits: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """Row-wise Euclidean projection onto the simplex (notebook 05 implementation)."""
    shifted = logits - logits.max(dim=dim, keepdim=True).values
    sorted_logits, _ = torch.sort(shifted, descending=True, dim=dim)
    range_values = torch.arange(1, logits.size(dim) + 1, device=logits.device, dtype=logits.dtype)
    view_shape = [1] * logits.dim()
    view_shape[dim] = -1
    range_values = range_values.view(view_shape)
    support = 1 + range_values * sorted_logits > torch.cumsum(sorted_logits, dim=dim)
    support_size = support.sum(dim=dim, keepdim=True).clamp_min(1)
    tau = (
        torch.gather(torch.cumsum(sorted_logits, dim=dim), dim, support_size - 1) - 1
    ) / support_size.to(logits.dtype)
    return torch.clamp(shifted - tau, min=0.0)


class HGDCTR(nn.Module):
    """EmerG x DGD hybrid.

    Kept from EmerG (notebook 04):
      * bounded dense sigmoid adjacency, so edge magnitude is preserved;
      * forced self-loops, so a field can never lose its own contribution;
      * additive residual message passing on the running field state h.

    Kept from DGD (notebook 05):
      * learnable simplex mixture of graph powers 1..l, so each layer chooses its
        own multi-hop receptive field;
      * row-wise sparsemax, so irrelevant field interactions are exactly zeroed;
      * multiplicative gating of the fields, computed from the layer-0 state.

    New, and aimed at the phase where both baselines are weakest (Cold):
      * the item id embedding is zero-initialised and always summed with an
        embedding synthesised from audited item side information, and id dropout
        is applied during training. An evaluation item is unseen by construction,
        so its id row is exactly the zero vector the model was trained to handle,
        rather than the untrained random vector EmerG and DGD score it with.

    At initialisation (alpha -> 0, modulation_gate -> 0, power mass on power 1,
    dense_temperature = sqrt(num_fields), synthesis head zeroed) the forward pass
    is exactly EmerG's update rule; `emerg_reduction_error` asserts this, and every
    added component is a gate the optimiser may leave closed.
    """

    def __init__(
        self,
        feature_sizes: dict[str, int],
        embedding_dim: int,
        hidden_dim: int,
        gnn_layers: int,
        sparsemax_scale: float,
        fusion_logit_init: float,
        modulation_gate_init: float,
        power_prior_bias: float,
        id_dropout: float,
    ):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_fields = len(FIELD_ORDER)
        self.gnn_layers = gnn_layers
        self.sparsemax_scale = float(sparsemax_scale)
        self.id_dropout = float(id_dropout)
        self.embeddings = nn.ModuleDict(
            {
                "user_id": nn.Embedding(feature_sizes["user_id"], embedding_dim),
                "gender": nn.Embedding(feature_sizes["gender"], embedding_dim, padding_idx=0),
                "age": nn.Embedding(feature_sizes["age"], embedding_dim, padding_idx=0),
                "occupation": nn.Embedding(feature_sizes["occupation"], embedding_dim, padding_idx=0),
                "zip_code": nn.Embedding(feature_sizes["zip_code"], embedding_dim, padding_idx=0),
                "item_id": nn.Embedding(feature_sizes["item_id"], embedding_dim),
                "genres": nn.Embedding(feature_sizes["genres"], embedding_dim, padding_idx=0),
                "title": nn.Embedding(feature_sizes["title"], embedding_dim, padding_idx=0),
            }
        )
        self.release_weight = nn.Parameter(torch.empty(embedding_dim))
        self.item_synth = nn.Sequential(
            nn.Linear(len(ITEM_SIDE_FIELDS) * embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embedding_dim),
        )
        self.item_graph_delta = nn.Embedding(feature_sizes["item_id"], self.num_fields * self.num_fields)
        self.graph_generator = nn.Sequential(
            nn.Linear(len(ITEM_GRAPH_FIELDS) * embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, self.num_fields * self.num_fields),
        )
        self.diffusion_logits = nn.Parameter(torch.zeros(gnn_layers, gnn_layers))
        self.dense_temperature = nn.Parameter(torch.empty(gnn_layers))
        self.fusion_logits = nn.Parameter(torch.empty(gnn_layers))
        self.modulation_gate = nn.Parameter(torch.empty(gnn_layers))
        self.graph_layers = nn.ModuleList(
            [nn.Linear(embedding_dim, embedding_dim, bias=False) for _ in range(gnn_layers)]
        )
        self.modulation_layers = nn.ModuleList(
            [nn.Linear(embedding_dim, embedding_dim, bias=False) for _ in range(gnn_layers)]
        )
        self.head = nn.Sequential(
            nn.Linear(self.num_fields * embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.register_buffer("identity_graph", torch.eye(self.num_fields))
        self.reset_parameters(fusion_logit_init, modulation_gate_init, power_prior_bias)

    def reset_parameters(
        self,
        fusion_logit_init: float,
        modulation_gate_init: float,
        power_prior_bias: float,
    ) -> None:
        for module in self.modules():
            if isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, std=0.02)
                if module.padding_idx is not None:
                    with torch.no_grad():
                        module.weight[module.padding_idx].zero_()
            elif isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
        nn.init.normal_(self.release_weight, std=0.02)
        nn.init.zeros_(self.item_graph_delta.weight)
        nn.init.zeros_(self.embeddings["item_id"].weight)
        with torch.no_grad():
            self.diffusion_logits.zero_()
            self.diffusion_logits[:, 0] = float(power_prior_bias)
            self.dense_temperature.fill_(float(self.num_fields) ** 0.5)
            self.fusion_logits.fill_(float(fusion_logit_init))
            self.modulation_gate.fill_(float(modulation_gate_init))

    def sequence_embedding(self, name: str, tokens: torch.Tensor) -> torch.Tensor:
        embedded = self.embeddings[name](tokens)
        mask = tokens.ne(0).float().unsqueeze(-1)
        denom = mask.sum(dim=1).clamp_min(1.0)
        return (embedded * mask).sum(dim=1) / denom

    def item_side_embeddings(self, features: dict[str, torch.Tensor]) -> list[torch.Tensor]:
        return [
            features["release_year"].unsqueeze(1) * self.release_weight.unsqueeze(0),
            self.sequence_embedding("genres", features["genres"]),
            self.sequence_embedding("title", features["title"]),
        ]

    def field_embeddings(
        self,
        features: dict[str, torch.Tensor],
        local_item_embedding: torch.Tensor | None = None,
    ) -> torch.Tensor:
        side = self.item_side_embeddings(features)
        synthetic_item = self.item_synth(torch.cat(side, dim=1))
        if local_item_embedding is None:
            id_item = self.embeddings["item_id"](features["item_id"])
            if self.training and self.id_dropout > 0.0:
                keep = (
                    torch.rand(id_item.shape[0], 1, device=id_item.device) >= self.id_dropout
                ).to(id_item.dtype)
                id_item = id_item * keep
        else:
            id_item = local_item_embedding.unsqueeze(0).expand(features["item_id"].shape[0], -1)
        fields = [
            self.embeddings["user_id"](features["user_id"]),
            self.embeddings["gender"](features["gender"]),
            self.embeddings["age"](features["age"]),
            self.embeddings["occupation"](features["occupation"]),
            self.embeddings["zip_code"](features["zip_code"]),
            id_item + synthetic_item,
            side[0],
            side[1],
            side[2],
        ]
        return torch.stack(fields, dim=1)

    def first_order_logits(
        self,
        features: dict[str, torch.Tensor],
        local_item_embedding: torch.Tensor | None = None,
        local_graph_delta: torch.Tensor | None = None,
    ) -> torch.Tensor:
        field_emb = self.field_embeddings(features, local_item_embedding)
        item_context = field_emb[:, ITEM_GRAPH_POSITIONS, :].reshape(field_emb.shape[0], -1)
        raw_graph = self.graph_generator(item_context)
        if local_graph_delta is None:
            raw_graph = raw_graph + self.item_graph_delta(features["item_id"])
        else:
            raw_graph = raw_graph + local_graph_delta.unsqueeze(0).expand_as(raw_graph)
        raw_graph = raw_graph.reshape(-1, self.num_fields, self.num_fields)
        return 0.5 * (raw_graph + raw_graph.transpose(1, 2))

    def fused_graphs(
        self,
        features: dict[str, torch.Tensor],
        local_item_embedding: torch.Tensor | None = None,
        local_graph_delta: torch.Tensor | None = None,
    ) -> tuple[list[torch.Tensor], list[torch.Tensor], torch.Tensor]:
        base_logits = self.first_order_logits(features, local_item_embedding, local_graph_delta)
        scale = float(self.num_fields) ** 0.5
        scaled = base_logits / scale
        powers = [scaled]
        for _ in range(1, self.gnn_layers):
            powers.append(torch.bmm(powers[-1], scaled) / scale)

        eye = self.identity_graph.unsqueeze(0)
        fused: list[torch.Tensor] = []
        sparse_branches: list[torch.Tensor] = []
        for layer_index in range(self.gnn_layers):
            weights = torch.softmax(self.diffusion_logits[layer_index, : layer_index + 1], dim=0)
            mixed = sum(
                weights[power_index] * powers[power_index]
                for power_index in range(layer_index + 1)
            )
            dense = torch.sigmoid(self.dense_temperature[layer_index] * mixed)
            sparse = sparsemax(self.sparsemax_scale * mixed, dim=-1)
            alpha = torch.sigmoid(self.fusion_logits[layer_index])
            graph = (1.0 - alpha) * dense + alpha * sparse
            graph = graph * (1.0 - eye) + eye
            fused.append(graph)
            sparse_branches.append(sparse)
        return fused, sparse_branches, base_logits

    def forward(
        self,
        features: dict[str, torch.Tensor],
        local_item_embedding: torch.Tensor | None = None,
        local_graph_delta: torch.Tensor | None = None,
    ) -> torch.Tensor:
        h0 = self.field_embeddings(features, local_item_embedding)
        graphs, _, _ = self.fused_graphs(features, local_item_embedding, local_graph_delta)
        h = h0
        for layer_index, graph in enumerate(graphs):
            additive = torch.bmm(graph, self.graph_layers[layer_index](h))
            h = h + F.relu(additive)
            modulation = torch.bmm(graph, self.modulation_layers[layer_index](h0))
            h = h * (1.0 + self.modulation_gate[layer_index] * torch.tanh(modulation))
        return self.head(h.reshape(h.shape[0], -1)).squeeze(1)

    def diffusion_weight_rows(self) -> list[dict[str, Any]]:
        rows: list[dict[str, Any]] = []
        for layer_index in range(self.gnn_layers):
            weights = torch.softmax(self.diffusion_logits[layer_index, : layer_index + 1], dim=0)
            for power_index, weight in enumerate(weights.detach().cpu().tolist(), start=1):
                rows.append({"layer": layer_index + 1, "power": power_index, "weight": float(weight)})
        return rows

    def gate_rows(self) -> list[dict[str, Any]]:
        rows: list[dict[str, Any]] = []
        for layer_index in range(self.gnn_layers):
            rows.append(
                {
                    "layer": layer_index + 1,
                    "sparse_fusion_alpha": float(
                        torch.sigmoid(self.fusion_logits[layer_index]).detach().cpu()
                    ),
                    "dense_temperature": float(self.dense_temperature[layer_index].detach().cpu()),
                    "modulation_gate": float(self.modulation_gate[layer_index].detach().cpu()),
                }
            )
        return rows


def emerg_reference_forward(model: HGDCTR, features: dict[str, torch.Tensor]) -> torch.Tensor:
    """Notebook-04 EmerG update rule evaluated with this model's parameters.

    Symmetrisation is applied in logit space (as in notebook 05) rather than after
    the sigmoid; everything else is EmerG's forward pass.
    """
    h = model.field_embeddings(features)
    graph = torch.sigmoid(model.first_order_logits(features))
    eye = model.identity_graph.unsqueeze(0)
    graph = graph * (1.0 - eye) + eye
    for layer in model.graph_layers:
        message = torch.bmm(graph, layer(h))
        h = h + F.relu(message)
    return model.head(h.reshape(h.shape[0], -1)).squeeze(1)


def emerg_reduction_error(model: HGDCTR, features: dict[str, torch.Tensor]) -> float:
    """Max |HGD - EmerG| once every added component is switched off."""
    reduced = copy.deepcopy(model).eval()
    with torch.no_grad():
        reduced.item_synth[-1].weight.zero_()
        reduced.item_synth[-1].bias.zero_()
        reduced.fusion_logits.fill_(-40.0)
        reduced.modulation_gate.zero_()
        reduced.diffusion_logits.fill_(-40.0)
        reduced.diffusion_logits[:, 0] = 40.0
        reduced.dense_temperature.fill_(float(reduced.num_fields) ** 0.5)
        error = (reduced(features) - emerg_reference_forward(reduced, features)).abs().max()
    return float(error.cpu())


def roc_auc(labels: np.ndarray, scores: np.ndarray) -> float:
    labels = labels.astype(np.int64)
    positives = int(labels.sum())
    negatives = int(len(labels) - positives)
    if positives == 0 or negatives == 0:
        return float("nan")
    order = np.argsort(scores, kind="mergesort")
    sorted_scores = scores[order]
    ranks = np.arange(1, len(scores) + 1, dtype=np.float64)
    start = 0
    while start < len(scores):
        end = start + 1
        while end < len(scores) and sorted_scores[end] == sorted_scores[start]:
            end += 1
        if end - start > 1:
            ranks[start:end] = ranks[start:end].mean()
        start = end
    original_ranks = np.empty_like(ranks)
    original_ranks[order] = ranks
    return float((original_ranks[labels == 1].sum() - positives * (positives + 1) / 2) / (positives * negatives))


def best_f1_threshold(labels: np.ndarray, scores: np.ndarray) -> tuple[float, float]:
    labels = labels.astype(np.int64)
    order = np.argsort(-scores, kind="mergesort")
    sorted_labels = labels[order]
    sorted_scores = scores[order]
    tp = np.cumsum(sorted_labels)
    fp = np.cumsum(1 - sorted_labels)
    fn = int(sorted_labels.sum()) - tp
    denominator = 2 * tp + fp + fn
    f1 = np.divide(2 * tp, denominator, out=np.zeros_like(tp, dtype=np.float64), where=denominator > 0)
    group_ends = np.flatnonzero(
        np.r_[sorted_scores[1:] != sorted_scores[:-1], True]
    )
    best = int(group_ends[np.argmax(f1[group_ends])])
    return float(sorted_scores[best]), float(f1[best])


def binary_metrics(labels: np.ndarray, scores: np.ndarray, threshold: float) -> dict[str, Any]:
    labels = labels.astype(np.int64)
    predictions = scores >= threshold
    tp = int(((predictions == 1) & (labels == 1)).sum())
    fp = int(((predictions == 1) & (labels == 0)).sum())
    fn = int(((predictions == 0) & (labels == 1)).sum())
    tn = int(((predictions == 0) & (labels == 0)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "rows": int(len(labels)),
        "positives": int(labels.sum()),
        "threshold": float(threshold),
        "accuracy": float((tp + tn) / len(labels)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "roc_auc": roc_auc(labels, scores),
        "predicted_positive_rate": float(predictions.mean()),
        "score_mean": float(scores.mean()),
        "score_std": float(scores.std()),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }


def summarize_predictions(
    predictions: pd.DataFrame,
    split: str,
    thresholds: dict[str, float] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    threshold_rows: list[dict[str, Any]] = []
    metric_rows: list[dict[str, Any]] = []
    for phase in PHASES:
        phase_predictions = predictions[predictions["phase"].eq(phase)]
        labels = phase_predictions["label"].to_numpy(dtype=np.int64)
        scores = phase_predictions["score"].to_numpy(dtype=np.float64)
        if thresholds is None:
            threshold, best_f1 = best_f1_threshold(labels, scores)
        else:
            threshold, best_f1 = float(thresholds[phase]), np.nan
        threshold_rows.append(
            {"split": split, "phase": phase, "threshold": threshold, "validation_best_f1": best_f1}
        )
        metric_rows.append({"split": split, "phase": phase, **binary_metrics(labels, scores, threshold)})
    return pd.DataFrame(threshold_rows), pd.DataFrame(metric_rows)


def numeric_tables_are_finite(*tables: pd.DataFrame) -> bool:
    for table in tables:
        numeric = table.select_dtypes(include=[np.number])
        if numeric.empty or not np.isfinite(numeric.to_numpy(dtype=np.float64)).all():
            return False
    return True

In [ ]:
def build_model(config: dict[str, Any]) -> HGDCTR:
    return HGDCTR(
        FEATURES.field_sizes,
        embedding_dim=int(config["embedding_dim"]),
        hidden_dim=int(config["hidden_dim"]),
        gnn_layers=int(config["gnn_layers"]),
        sparsemax_scale=float(config["sparsemax_scale"]),
        fusion_logit_init=float(config["fusion_logit_init"]),
        modulation_gate_init=float(config["modulation_gate_init"]),
        power_prior_bias=float(config["power_prior_bias"]),
        id_dropout=float(config["id_dropout"]),
    ).to(DEVICE)


def excited_probe(model: HGDCTR, seed: int) -> HGDCTR:
    """Copy of the model moved into a trained-like graph regime.

    At initialisation the generated logits are ~0, so every graph is uniform and
    the mixing/fusion gradients are numerically indistinguishable from zero. The
    probe perturbs only the graph read-outs, which is the regime the gradient
    checks are meant to interrogate.
    """
    probe = copy.deepcopy(model).train()
    generator = torch.Generator(device="cpu").manual_seed(seed)
    with torch.no_grad():
        for parameter in (
            probe.graph_generator[-1].bias,
            probe.item_graph_delta.weight,
            probe.embeddings["item_id"].weight,
        ):
            parameter.copy_(
                torch.randn(parameter.shape, generator=generator).to(parameter.device) * 0.5
            )
    return probe


def hgd_unit_and_gradient_check(run_config: dict[str, Any], seed: int) -> dict[str, Any]:
    config = dict(run_config["candidate_configs"][0])
    model = build_model(config)
    arrays = FEATURES.arrays(TUNING_TRAIN.iloc[: min(1024, len(TUNING_TRAIN))])
    rows = np.arange(len(arrays["label"]), dtype=np.int64)
    features, labels = FEATURES.batch(arrays, rows)

    reduction_error = emerg_reduction_error(model, features)
    cold_id_zero = float(model.embeddings["item_id"].weight.detach().abs().max().cpu())

    probe = excited_probe(model, seed)
    logits = probe(features)
    graphs, sparse_branches, base = probe.fused_graphs(features)
    loss = F.binary_cross_entropy_with_logits(logits, labels) + sum(
        graph.mean() for graph in graphs
    ) * 1e-3
    probe.zero_grad(set_to_none=True)
    loss.backward()

    def grad_sum(parameters: Iterable[nn.Parameter]) -> float:
        total = 0.0
        for parameter in parameters:
            if parameter.grad is not None:
                total += float(parameter.grad.detach().abs().sum().cpu())
        return total

    graph_grad_norm = grad_sum(probe.graph_generator.parameters())
    diffusion_grad_norm = grad_sum([probe.diffusion_logits])
    fusion_grad_norm = grad_sum([probe.fusion_logits])
    modulation_grad_norm = grad_sum([probe.modulation_gate]) + grad_sum(
        probe.modulation_layers.parameters()
    )
    synthesis_grad_norm = grad_sum(probe.item_synth.parameters())
    id_grad_norm = grad_sum([probe.embeddings["item_id"].weight])

    sparse_fixture = sparsemax(torch.tensor([[3.0, 1.0, -2.0]], device=DEVICE), dim=-1)
    fixture_sparse_fraction = float((sparse_fixture <= 1e-6).float().mean().detach().cpu())
    row_sums = torch.cat([branch.sum(dim=-1).detach().flatten() for branch in sparse_branches])
    row_sum_error = float((row_sums - 1.0).abs().max().cpu())
    sparse_fraction = float(
        torch.cat([(branch <= 1e-6).float().detach().flatten() for branch in sparse_branches])
        .mean()
        .cpu()
    )
    diagonals = torch.stack([graph.diagonal(dim1=1, dim2=2).detach() for graph in graphs])
    self_loop_error = float((diagonals - 1.0).abs().max().cpu())
    finite_values = bool(
        torch.isfinite(base).all() and all(torch.isfinite(graph).all() for graph in graphs)
    )

    result = {
        "loss": float(loss.detach().cpu()),
        "emerg_reduction_max_abs_error": reduction_error,
        "cold_id_embedding_max_abs": cold_id_zero,
        "graph_generator_grad_norm": graph_grad_norm,
        "diffusion_grad_norm": diffusion_grad_norm,
        "fusion_grad_norm": fusion_grad_norm,
        "modulation_grad_norm": modulation_grad_norm,
        "synthesis_grad_norm": synthesis_grad_norm,
        "item_id_grad_norm": id_grad_norm,
        "sparsemax_row_sum_error": row_sum_error,
        "sparse_fraction": sparse_fraction,
        "fixture_sparse_fraction": fixture_sparse_fraction,
        "self_loop_max_abs_error": self_loop_error,
        "finite_values": finite_values,
    }
    result["status"] = (
        "PASS"
        if reduction_error < 1e-4
        and cold_id_zero == 0.0
        and graph_grad_norm > 0
        and diffusion_grad_norm > 0
        and fusion_grad_norm > 0
        and modulation_grad_norm > 0
        and synthesis_grad_norm > 0
        and id_grad_norm > 0
        and row_sum_error < 1e-4
        and sparse_fraction > 0
        and fixture_sparse_fraction > 0
        and self_loop_error < 1e-6
        and finite_values
        else "FAIL"
    )
    del model, probe
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return result


HGD_UNIT_CHECK = hgd_unit_and_gradient_check(make_run_config(TARGET_SEEDS[0]), TARGET_SEEDS[0])
show_records([HGD_UNIT_CHECK])
display(
    Markdown(
        "`emerg_reduction_max_abs_error` is the max absolute score difference between "
        "HGD with every added component switched off and the EmerG update rule on the "
        "same parameters. It is the evidence that HGD contains EmerG as a special case, "
        "so the added gates can only be exercised when validation asks for them."
    )
)
if HGD_UNIT_CHECK["status"] != "PASS":
    raise RuntimeError("HGD unit/gradient checks failed; do not train from this state")


In [ ]:
def train_hgd_model(
    train_table: pd.DataFrame,
    config: dict[str, Any],
    seed: int,
    run_name: str,
) -> tuple[HGDCTR, pd.DataFrame, float]:
    start = time.perf_counter()
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    arrays = FEATURES.arrays(train_table)
    model = build_model(config)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=float(config["learning_rate"]), weight_decay=float(config["weight_decay"])
    )
    history: list[dict[str, Any]] = []
    n_rows = len(arrays["label"])
    sample_rows = min(int(config["epoch_sample_rows"]), n_rows)
    batch_size = int(config["batch_size"])

    for epoch in range(1, int(config["epochs"]) + 1):
        model.train()
        order = rng.choice(n_rows, size=sample_rows, replace=False) if sample_rows < n_rows else rng.permutation(n_rows)
        total_loss = 0.0
        total_examples = 0
        for start_idx in range(0, len(order), batch_size):
            rows = order[start_idx : start_idx + batch_size]
            features, labels = FEATURES.batch(arrays, rows)
            optimizer.zero_grad(set_to_none=True)
            logits = model(features)
            loss = F.binary_cross_entropy_with_logits(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.detach().cpu()) * len(rows)
            total_examples += len(rows)
        history.append(
            {
                "run_name": run_name,
                "epoch": epoch,
                "loss": total_loss / max(total_examples, 1),
                "examples": total_examples,
            }
        )
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return model, pd.DataFrame(history), time.perf_counter() - start


def freeze_parameters(model: nn.Module, value: bool) -> list[bool]:
    previous = [parameter.requires_grad for parameter in model.parameters()]
    for parameter in model.parameters():
        parameter.requires_grad_(value)
    return previous


def restore_requires_grad(model: nn.Module, previous: list[bool]) -> None:
    for parameter, requires_grad in zip(model.parameters(), previous):
        parameter.requires_grad_(requires_grad)


def predict_table(
    model: HGDCTR,
    table: pd.DataFrame,
    local_item_embedding: torch.Tensor | None = None,
    local_graph_delta: torch.Tensor | None = None,
    batch_size: int = 8192,
) -> np.ndarray:
    arrays = FEATURES.arrays(table)
    scores: list[np.ndarray] = []
    model.eval()
    with torch.no_grad():
        for start_idx in range(0, len(table), batch_size):
            rows = np.arange(start_idx, min(start_idx + batch_size, len(table)), dtype=np.int64)
            features, _ = FEATURES.batch(arrays, rows)
            logits = model(features, local_item_embedding, local_graph_delta)
            scores.append(torch.sigmoid(logits).detach().cpu().numpy())
    return np.concatenate(scores).astype(np.float32)


def adapt_local_state(
    model: HGDCTR,
    support_table: pd.DataFrame,
    local_item_embedding: torch.Tensor,
    local_graph_delta: torch.Tensor,
    config: dict[str, Any],
) -> tuple[torch.Tensor, torch.Tensor, float]:
    if support_table.empty:
        return local_item_embedding, local_graph_delta, 0.0
    arrays = FEATURES.arrays(support_table.reset_index(drop=True))
    rows = np.arange(len(support_table), dtype=np.int64)
    optimizer = torch.optim.Adam([local_item_embedding, local_graph_delta], lr=float(config["warm_learning_rate"]))
    last_loss = 0.0
    model.eval()
    for _ in range(int(config["warm_steps"])):
        features, labels = FEATURES.batch(arrays, rows)
        optimizer.zero_grad(set_to_none=True)
        logits = model(features, local_item_embedding, local_graph_delta)
        loss = F.binary_cross_entropy_with_logits(logits, labels)
        loss.backward()
        optimizer.step()
        last_loss = float(loss.detach().cpu())
    return local_item_embedding.detach().requires_grad_(), local_graph_delta.detach().requires_grad_(), last_loss


def graph_stats(
    model: HGDCTR,
    one_row: pd.DataFrame,
    phase: str,
    local_item_embedding: torch.Tensor | None,
    local_graph_delta: torch.Tensor | None,
    local_loss: float,
) -> dict[str, Any]:
    arrays = FEATURES.arrays(one_row.reset_index(drop=True))
    features, _ = FEATURES.batch(arrays, np.array([0], dtype=np.int64))
    with torch.no_grad():
        graphs, sparse_branches, base = model.fused_graphs(
            features, local_item_embedding, local_graph_delta
        )
        final_graph = graphs[-1][0].detach().cpu().numpy()
        final_sparse = sparse_branches[-1][0].detach().cpu().numpy()
        base_graph = base[0].detach().cpu().numpy()
    return {
        "item_id": int(one_row["item_id"].iloc[0]),
        "item_idx": int(one_row["item_idx"].iloc[0]),
        "phase": phase,
        "base_graph_mean": float(base_graph.mean()),
        "fused_graph_mean": float(final_graph.mean()),
        "fused_graph_std": float(final_graph.std()),
        "fused_density_gt_1e_6": float((final_graph > 1e-6).mean()),
        "fused_diag_mean": float(np.diag(final_graph).mean()),
        "sparse_branch_density_gt_1e_6": float((final_sparse > 1e-6).mean()),
        "sparse_branch_row_sum_error": float(np.abs(final_sparse.sum(axis=1) - 1.0).max()),
        "local_delta_norm": float(local_graph_delta.detach().norm().cpu())
        if local_graph_delta is not None
        else 0.0,
        "last_warm_loss": float(local_loss),
    }


def score_tasks_with_warmup(
    model: HGDCTR,
    tasks: pd.DataFrame,
    config: dict[str, Any],
    split: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    predictions: list[pd.DataFrame] = []
    diagnostics: list[dict[str, Any]] = []
    previous_requires_grad = freeze_parameters(model, False)
    try:
        sorted_tasks = tasks.sort_values(["item_idx", "item_rank", "source_row"]).reset_index(drop=True)
        for item_idx, item_rows in sorted_tasks.groupby("item_idx", sort=True):
            item_rows = item_rows.reset_index(drop=True)
            query_rows = item_rows[item_rows["role"].eq("query")].reset_index(drop=True)
            base_embedding = model.embeddings["item_id"].weight[int(item_idx)].detach().clone().requires_grad_()
            base_delta = model.item_graph_delta.weight[int(item_idx)].detach().clone().requires_grad_()

            cold_scores = predict_table(model, query_rows)
            cold_predictions = query_rows[
                ["source_row", "user_id", "user_idx", "item_id", "item_idx", "label"]
            ].copy()
            cold_predictions.insert(0, "phase", "Cold")
            cold_predictions["score"] = cold_scores
            predictions.append(cold_predictions)
            diagnostics.append(graph_stats(model, query_rows.iloc[[0]], "Cold", None, None, 0.0))

            local_embedding = base_embedding
            local_delta = base_delta
            for phase, role in PHASE_INCREMENT_ROLE.items():
                support_rows = item_rows[item_rows["role"].eq(role)].reset_index(drop=True)
                local_embedding, local_delta, warm_loss = adapt_local_state(
                    model, support_rows, local_embedding, local_delta, config
                )
                phase_scores = predict_table(model, query_rows, local_embedding, local_delta)
                phase_predictions = query_rows[
                    ["source_row", "user_id", "user_idx", "item_id", "item_idx", "label"]
                ].copy()
                phase_predictions.insert(0, "phase", phase)
                phase_predictions["score"] = phase_scores
                predictions.append(phase_predictions)
                diagnostics.append(
                    graph_stats(
                        model, query_rows.iloc[[0]], phase, local_embedding, local_delta, warm_loss
                    )
                )
    finally:
        restore_requires_grad(model, previous_requires_grad)
    prediction_table = pd.concat(predictions, ignore_index=True)
    diagnostics_table = pd.DataFrame(diagnostics)
    diagnostics_table.insert(0, "split", split)
    return prediction_table, diagnostics_table

In [ ]:
def json_ready(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_ready(item) for item in value]
    if hasattr(value, "item"):
        return value.item()
    return str(value)


def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temporary.write_text(content, encoding="utf-8")
    temporary.replace(path)


def write_csv(path: Path, table: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    table.to_csv(temporary, index=False, lineterminator="\n")
    temporary.replace(path)


def relative_output(path: Path) -> str:
    return str(path.resolve().relative_to(ARTIFACT_ROOT.resolve()))


def csv_schema(table: pd.DataFrame) -> dict[str, Any]:
    def dtype_name(dtype: Any) -> str:
        name = str(dtype)
        return "string" if name in {"str", "string"} or name.startswith("string") else name

    return {
        "columns": list(table.columns),
        "read_csv_dtypes": {column: dtype_name(dtype) for column, dtype in table.dtypes.items()},
    }


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def run_hgd_seed(seed: int, bundle_id: str) -> dict[str, Any]:
    run_started = time.perf_counter()
    run_config = make_run_config(seed)
    run_config_hash = hashlib.sha256(
        json.dumps(
            run_config, sort_keys=True, separators=(",", ":"), allow_nan=False
        ).encode()
    ).hexdigest()
    seed_everything(seed)

    hgd_check = hgd_unit_and_gradient_check(run_config, seed)
    if hgd_check["status"] != "PASS":
        display(pd.DataFrame([hgd_check]))
        raise RuntimeError(f"HGD unit/gradient check failed for seed {seed}")

    training_history_parts: list[pd.DataFrame] = []
    config_summary_rows: list[dict[str, Any]] = []
    selected_payload: dict[str, Any] | None = None

    for config_index, config in enumerate(run_config["candidate_configs"], start=1):
        config_name = f"cfg{config_index:02d}_G{config['gnn_layers']}_D{config['embedding_dim']}"
        model, history, training_seconds = train_hgd_model(
            TUNING_TRAIN, config, seed=seed + config_index, run_name=config_name
        )
        history["config_name"] = config_name
        training_history_parts.append(history)

        validation_predictions, validation_diagnostics = score_tasks_with_warmup(
            model, VALIDATION_TASKS, config, split="validation"
        )
        thresholds, metrics = summarize_predictions(validation_predictions, split="validation")
        thresholds["config_name"] = config_name
        metrics["config_name"] = config_name
        validation_diagnostics["config_name"] = config_name

        mean_f1 = float(metrics["f1"].mean())
        mean_auc = float(metrics["roc_auc"].mean())
        summary = {
            "config_name": config_name,
            "mean_validation_f1": mean_f1,
            "mean_validation_auc": mean_auc,
            "training_seconds": float(training_seconds),
            **config,
        }
        config_summary_rows.append(summary)
        candidate_payload = {
            "config_name": config_name,
            "config": dict(config),
            "summary": summary,
            "thresholds": thresholds,
            "metrics": metrics,
            "predictions": validation_predictions,
            "diagnostics": validation_diagnostics,
        }
        if selected_payload is None or (mean_f1, mean_auc) > (
            selected_payload["summary"]["mean_validation_f1"],
            selected_payload["summary"]["mean_validation_auc"],
        ):
            selected_payload = candidate_payload
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    if selected_payload is None:
        raise RuntimeError(f"No HGD candidate was trained for seed {seed}")

    training_history = pd.concat(training_history_parts, ignore_index=True)
    config_summary = pd.DataFrame(config_summary_rows).sort_values(
        ["mean_validation_f1", "mean_validation_auc"], ascending=False
    )
    selected_config_name = str(selected_payload["config_name"])
    selected_config = dict(selected_payload["config"])
    selected_summary = dict(selected_payload["summary"])
    validation_predictions = selected_payload["predictions"].copy()
    validation_thresholds = selected_payload["thresholds"].copy()
    validation_metrics = selected_payload["metrics"].copy()
    validation_diagnostics = selected_payload["diagnostics"].copy()
    threshold_by_phase = dict(
        zip(validation_thresholds["phase"], validation_thresholds["threshold"])
    )

    final_model, final_history, final_training_seconds = train_hgd_model(
        FINAL_TRAIN,
        selected_config,
        seed=seed + 10_000,
        run_name=f"final_refit_{selected_config_name}",
    )
    final_history["config_name"] = selected_config_name
    training_history = pd.concat([training_history, final_history], ignore_index=True)

    evaluation_predictions, evaluation_diagnostics = score_tasks_with_warmup(
        final_model, EVALUATION_TASKS, selected_config, split="evaluation"
    )
    _, evaluation_metrics = summarize_predictions(
        evaluation_predictions, split="evaluation", thresholds=threshold_by_phase
    )
    evaluation_metrics["config_name"] = selected_config_name
    graph_diagnostics = pd.concat(
        [validation_diagnostics, evaluation_diagnostics], ignore_index=True
    )
    graph_diagnostics["selected_config_name"] = selected_config_name
    graph_diagnostics["final_training_seconds"] = float(final_training_seconds)
    graph_diagnostics["device"] = str(DEVICE)
    fusion_gates = pd.DataFrame(final_model.gate_rows())
    diffusion_weights = pd.DataFrame(final_model.diffusion_weight_rows())

    pre_export_checks: list[dict[str, Any]] = []

    def pre_export_check(name: str, condition: bool, observed: Any, expected: Any) -> None:
        pre_export_checks.append(
            {
                "check": name,
                "status": "PASS" if condition else "FAIL",
                "observed": observed,
                "expected": expected,
            }
        )

    expected_validation_predictions = int(
        VALIDATION_TASKS["role"].eq("query").sum() * len(PHASES)
    )
    expected_evaluation_predictions = int(
        EVALUATION_TASKS["role"].eq("query").sum() * len(PHASES)
    )
    final_excludes_evaluation_items = set(FINAL_TRAIN["item_idx"]).isdisjoint(
        set(EVALUATION_TASKS["item_idx"])
    )
    feature_contract_has_pad_unk = all(
        vocab.get(PAD) == 0 and vocab.get(UNK) == 1
        for vocab in [
            FEATURES.gender_vocab,
            FEATURES.age_vocab,
            FEATURES.occupation_vocab,
            FEATURES.zip_vocab,
            FEATURES.genre_vocab,
            FEATURES.title_vocab,
        ]
    )
    diffusion_rows_expected = sum(range(1, int(selected_config["gnn_layers"]) + 1))
    diffusion_rows_valid = len(diffusion_weights) == diffusion_rows_expected and np.isfinite(
        diffusion_weights["weight"]
    ).all()
    fusion_gates_valid = bool(
        len(fusion_gates) == int(selected_config["gnn_layers"])
        and np.isfinite(fusion_gates[["sparse_fusion_alpha", "dense_temperature", "modulation_gate"]].to_numpy()).all()
        and fusion_gates["sparse_fusion_alpha"].between(0.0, 1.0).all()
    )
    training_values_finite = numeric_tables_are_finite(training_history)
    validation_values_finite = numeric_tables_are_finite(
        config_summary,
        validation_thresholds,
        validation_metrics,
        validation_predictions,
    )
    evaluation_values_finite = numeric_tables_are_finite(
        evaluation_predictions, evaluation_metrics
    )

    pre_export_check("run config seed", run_config["seed"] == seed, run_config["seed"], seed)
    pre_export_check(
        "protocol schema",
        PROTOCOL_MANIFEST["protocol_schema_version"] == "ml1m-coldstart-v1",
        PROTOCOL_MANIFEST["protocol_schema_version"],
        "ml1m-coldstart-v1",
    )
    pre_export_check("hgd gradient check", hgd_check["status"] == "PASS", hgd_check["status"], "PASS")
    pre_export_check(
        "hgd reduces to emerg at the closed-gate point",
        hgd_check["emerg_reduction_max_abs_error"] < 1e-4,
        hgd_check["emerg_reduction_max_abs_error"],
        "< 1e-4",
    )
    pre_export_check(
        "cold item id embedding is the trained-for zero vector",
        hgd_check["cold_id_embedding_max_abs"] == 0.0,
        hgd_check["cold_id_embedding_max_abs"],
        0.0,
    )
    pre_export_check(
        "phase thresholds complete",
        set(threshold_by_phase) == set(PHASES),
        sorted(threshold_by_phase),
        sorted(PHASES),
    )
    pre_export_check(
        "validation predictions complete",
        len(validation_predictions) == expected_validation_predictions,
        len(validation_predictions),
        expected_validation_predictions,
    )
    pre_export_check(
        "evaluation predictions complete",
        len(evaluation_predictions) == expected_evaluation_predictions,
        len(evaluation_predictions),
        expected_evaluation_predictions,
    )
    pre_export_check(
        "evaluation phases complete",
        set(evaluation_metrics["phase"]) == set(PHASES),
        sorted(evaluation_metrics["phase"]),
        sorted(PHASES),
    )
    pre_export_check(
        "final train excludes evaluation items",
        final_excludes_evaluation_items,
        final_excludes_evaluation_items,
        True,
    )
    pre_export_check(
        "feature contract has PAD/UNK",
        feature_contract_has_pad_unk,
        feature_contract_has_pad_unk,
        True,
    )
    pre_export_check(
        "graph diagnostics complete",
        set(graph_diagnostics["phase"]) == set(PHASES),
        sorted(graph_diagnostics["phase"].unique()),
        sorted(PHASES),
    )
    pre_export_check(
        "self loops preserved on every scored graph",
        float(np.abs(graph_diagnostics["fused_diag_mean"] - 1.0).max()) < 1e-6,
        float(np.abs(graph_diagnostics["fused_diag_mean"] - 1.0).max()),
        "< 1e-6",
    )
    pre_export_check(
        "fusion gates valid",
        fusion_gates_valid,
        len(fusion_gates),
        int(selected_config["gnn_layers"]),
    )
    pre_export_check(
        "diffusion weights valid",
        diffusion_rows_valid,
        len(diffusion_weights),
        diffusion_rows_expected,
    )
    pre_export_check(
        "training values finite",
        training_values_finite,
        training_values_finite,
        True,
    )
    pre_export_check(
        "validation values finite",
        validation_values_finite,
        validation_values_finite,
        True,
    )
    pre_export_check(
        "evaluation values finite",
        evaluation_values_finite,
        evaluation_values_finite,
        True,
    )
    if not all(row["status"] == "PASS" for row in pre_export_checks):
        display(pd.DataFrame(pre_export_checks))
        raise RuntimeError(f"HGD semantic checks failed for seed {seed}; artifacts were not published")

    output_tables = {
        "training_history": training_history,
        "config_summary": config_summary,
        "validation_thresholds": validation_thresholds,
        "validation_metrics": validation_metrics,
        "validation_predictions": validation_predictions,
        "evaluation_metrics": evaluation_metrics,
        "evaluation_predictions": evaluation_predictions,
        "graph_diagnostics": graph_diagnostics,
        "diffusion_weights": diffusion_weights,
        "fusion_gates": fusion_gates,
    }
    staging_root = MODEL_OUTPUT_ROOT / f".staging-{bundle_id}"
    generation_root = MODEL_OUTPUT_ROOT / "generations" / bundle_id
    pointer_path = MODEL_OUTPUT_ROOT / "manifest.json"
    staging_root.mkdir(parents=True, exist_ok=False)
    output_artifacts: dict[str, Any] = {}
    previous_pointer_text = (
        pointer_path.read_text(encoding="utf-8") if pointer_path.is_file() else None
    )
    generation_published = False
    pointer_write_attempted = False

    selected_config_payload = {
        "schema_version": run_config["schema_version"],
        "selected_config_name": selected_config_name,
        "selected_config": selected_config,
        "selected_summary": selected_summary,
        "threshold_by_phase": threshold_by_phase,
        "run_config_sha256": run_config_hash,
        "protocol_pointer_sha256": PROTOCOL_POINTER_SHA256,
    }

    try:
        for name, table in output_tables.items():
            staging_path = staging_root / f"{name}.csv"
            published_path = generation_root / f"{name}.csv"
            write_csv(staging_path, table)
            output_artifacts[name] = {
                "path": relative_output(published_path),
                "sha256": sha256_file(staging_path),
                "rows": len(table),
            }

        selected_config_path = staging_root / "selected_config.json"
        write_text(
            selected_config_path,
            json.dumps(
                json_ready(selected_config_payload),
                indent=2,
                sort_keys=True,
                allow_nan=False,
            )
            + "\n",
        )
        output_artifacts["selected_config"] = {
            "path": relative_output(generation_root / "selected_config.json"),
            "sha256": sha256_file(selected_config_path),
        }

        feature_contract_path = staging_root / "feature_contract.json"
        write_text(
            feature_contract_path,
            json.dumps(
                json_ready(FEATURE_CONTRACT),
                indent=2,
                sort_keys=True,
                allow_nan=False,
            )
            + "\n",
        )
        output_artifacts["feature_contract"] = {
            "path": relative_output(generation_root / "feature_contract.json"),
            "sha256": sha256_file(feature_contract_path),
        }

        checkpoint_path = staging_root / "final_model.pt"
        torch.save(
            {
                "schema_version": run_config["schema_version"],
                "model_state_dict": final_model.state_dict(),
                "selected_config": selected_config,
                "feature_contract": FEATURE_CONTRACT,
                "threshold_by_phase": threshold_by_phase,
                "n_users": N_USERS,
                "n_items": N_ITEMS,
                "protocol_pointer_sha256": PROTOCOL_POINTER_SHA256,
            },
            checkpoint_path,
        )
        output_artifacts["final_model_checkpoint"] = {
            "path": relative_output(generation_root / "final_model.pt"),
            "sha256": sha256_file(checkpoint_path),
        }

        manifest = {
            "model_schema_version": run_config["schema_version"],
            "model_status": "PASS",
            "model_name": "HGD",
            "bundle_id": bundle_id,
            "bundle_manifest": relative_output(generation_root / "manifest.json"),
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "upstream_protocol": {
                "schema_version": PROTOCOL_MANIFEST["protocol_schema_version"],
                "bundle_id": PROTOCOL_MANIFEST["bundle_id"],
                "pointer_sha256": PROTOCOL_POINTER_SHA256,
            },
            "run_config": run_config,
            "run_config_sha256": run_config_hash,
            "feature_contract_sha256": sha256_file(feature_contract_path),
            "selected_config": selected_config_payload,
            "checks": pre_export_checks,
            "summary": {
                "selected_config_name": selected_config_name,
                "mean_validation_f1": selected_summary["mean_validation_f1"],
                "mean_validation_auc": selected_summary["mean_validation_auc"],
                "mean_evaluation_f1": float(evaluation_metrics["f1"].mean()),
                "mean_evaluation_auc": float(evaluation_metrics["roc_auc"].mean()),
                "evaluation_prediction_rows": len(evaluation_predictions),
                "final_training_seconds": float(final_training_seconds),
            },
            "artifacts": output_artifacts,
            "output_schemas": {
                name: csv_schema(table) for name, table in output_tables.items()
            },
            "phase_contract": PROTOCOL_MANIFEST["phase_contract"],
            "training_contract": {
                "tuning_base": "tuning_train from notebook 02",
                "final_refit_base": "final_train from notebook 02 after thresholds/config are frozen",
                "features": "audited user/item side information only; no interaction_count/cohort/timestamp/rank features",
                "graph": run_config["graph"],
                "propagation": run_config["propagation"],
                "cold_start_prior": run_config["cold_start_prior"],
                "budget_parity": run_config["budget_parity"],
                "warmup": run_config["warmup_policy"],
                "thresholds": "selected on validation query rows only and frozen for evaluation",
            },
        }

        manifest_text = (
            json.dumps(
                json_ready(manifest), indent=2, sort_keys=True, allow_nan=False
            )
            + "\n"
        )
        write_text(staging_root / "manifest.json", manifest_text)
        generation_root.parent.mkdir(parents=True, exist_ok=True)
        staging_root.replace(generation_root)
        generation_published = True
        generation_hashes_match = all(
            sha256_file(resolve_inside(ARTIFACT_ROOT, artifact["path"])) == artifact["sha256"]
            for artifact in output_artifacts.values()
        )
        generation_manifest_matches = (
            generation_root / "manifest.json"
        ).read_text(encoding="utf-8") == manifest_text
        if not generation_hashes_match or not generation_manifest_matches:
            raise RuntimeError(f"Published HGD generation failed verification for seed {seed}")
        pointer_write_attempted = True
        write_text(pointer_path, manifest_text)
    except Exception:
        try:
            if pointer_write_attempted:
                if previous_pointer_text is None:
                    pointer_path.unlink(missing_ok=True)
                else:
                    write_text(pointer_path, previous_pointer_text)
        finally:
            if staging_root.exists():
                shutil.rmtree(staging_root, ignore_errors=True)
            if generation_published and generation_root.exists():
                shutil.rmtree(generation_root)
        raise

    export_checks = list(pre_export_checks)

    def export_check(name: str, condition: bool, observed: Any, expected: Any) -> None:
        export_checks.append(
            {
                "check": name,
                "status": "PASS" if condition else "FAIL",
                "observed": observed,
                "expected": expected,
            }
        )

    try:
        manifest_pointer_matches = pointer_path.read_text(encoding="utf-8") == (
            generation_root / "manifest.json"
        ).read_text(encoding="utf-8")
        artifact_hashes_match = all(
            sha256_file(resolve_inside(ARTIFACT_ROOT, artifact["path"]))
            == artifact["sha256"]
            for artifact in manifest["artifacts"].values()
        )
        checkpoint_exported = "final_model_checkpoint" in manifest["artifacts"]
        feature_contract_exported = "feature_contract" in manifest["artifacts"]
        diffusion_weights_exported = "diffusion_weights" in manifest["artifacts"]
        fusion_gates_exported = "fusion_gates" in manifest["artifacts"]

        export_check(
            "manifest pointer matches bundle",
            manifest_pointer_matches,
            manifest_pointer_matches,
            True,
        )
        export_check(
            "artifact hashes verify", artifact_hashes_match, artifact_hashes_match, True
        )
        export_check("checkpoint exported", checkpoint_exported, checkpoint_exported, True)
        export_check(
            "feature contract exported",
            feature_contract_exported,
            feature_contract_exported,
            True,
        )
        export_check(
            "diffusion weights exported",
            diffusion_weights_exported,
            diffusion_weights_exported,
            True,
        )
        export_check(
            "fusion gates exported", fusion_gates_exported, fusion_gates_exported, True
        )
        if not all(row["status"] == "PASS" for row in export_checks):
            raise RuntimeError(f"HGD export checks failed for seed {seed}")
    except Exception:
        try:
            if previous_pointer_text is None:
                pointer_path.unlink(missing_ok=True)
            else:
                write_text(pointer_path, previous_pointer_text)
        finally:
            if generation_root.exists():
                shutil.rmtree(generation_root)
        raise

    result = {
        "seed": seed,
        "status": "PASS",
        "bundle_id": bundle_id,
        "generation_root": generation_root,
        "pointer_path": pointer_path,
        "manifest": manifest,
        "manifest_text": manifest_text,
        "export_checks": export_checks,
        "selected_config_name": selected_config_name,
        "evaluation_metrics": evaluation_metrics,
        "diffusion_weights": diffusion_weights,
        "fusion_gates": fusion_gates,
        "unit_check": hgd_check,
        "elapsed_seconds": time.perf_counter() - run_started,
    }
    del final_model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return result

In [ ]:
RUN_RESULTS: list[dict[str, Any]] = []
RUN_REGISTRY_ROWS: list[dict[str, Any]] = []
RUN_FAILURES: list[str] = []

for target_seed in TARGET_SEEDS:
    seed_started = time.perf_counter()
    planned_bundle_id = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
        + f"-s{target_seed}-"
        + uuid.uuid4().hex[:12]
    )
    try:
        run_result = run_hgd_seed(target_seed, planned_bundle_id)
    except Exception as error:
        failed_generation = MODEL_OUTPUT_ROOT / "generations" / planned_bundle_id
        generation_published = (failed_generation / "manifest.json").is_file()
        artifact_count = 0
        if generation_published:
            try:
                artifact_count = len(
                    json.loads(
                        (failed_generation / "manifest.json").read_text(encoding="utf-8")
                    ).get("artifacts", {})
                )
            except Exception:
                artifact_count = 0
        RUN_FAILURES.append(f"seed {target_seed}: {type(error).__name__}: {error}")
        RUN_REGISTRY_ROWS.append(
            {
                "seed": target_seed,
                "status": "BLOCKED",
                "bundle_id": planned_bundle_id if generation_published else None,
                "selected_config_name": None,
                "artifacts": artifact_count,
                "elapsed_seconds": round(time.perf_counter() - seed_started, 3),
                "error": f"{type(error).__name__}: {error}",
            }
        )
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    else:
        RUN_RESULTS.append(run_result)
        RUN_REGISTRY_ROWS.append(
            {
                "seed": target_seed,
                "status": "PASS",
                "bundle_id": run_result["bundle_id"],
                "selected_config_name": run_result["selected_config_name"],
                "artifacts": len(run_result["manifest"]["artifacts"]),
                "elapsed_seconds": round(run_result["elapsed_seconds"], 3),
                "error": None,
            }
        )

RUN_REGISTRY = pd.DataFrame(RUN_REGISTRY_ROWS)
display(RUN_REGISTRY)

In [ ]:
def verify_generation(result: dict[str, Any]) -> dict[str, Any]:
    verification = {
        "seed": result["seed"],
        "seed_matches": False,
        "run_config_hash_matches": False,
        "bundle_matches": False,
        "artifact_hashes_match": False,
        "error": None,
    }
    try:
        manifest_path = resolve_inside(ARTIFACT_ROOT, result["manifest"]["bundle_manifest"])
        generation_text = manifest_path.read_text(encoding="utf-8")
        generation_manifest = json.loads(generation_text)
        generation_run_config = generation_manifest.get("run_config", {})
        actual_run_config_hash = hashlib.sha256(
            json.dumps(
                generation_run_config,
                sort_keys=True,
                separators=(",", ":"),
                allow_nan=False,
            ).encode()
        ).hexdigest()
        verification.update(
            {
                "seed_matches": generation_run_config.get("seed") == result["seed"],
                "run_config_hash_matches": generation_manifest.get("run_config_sha256")
                == actual_run_config_hash,
                "bundle_matches": generation_manifest.get("bundle_id") == result["bundle_id"]
                and generation_text == result["manifest_text"],
                "artifact_hashes_match": all(
                    sha256_file(resolve_inside(ARTIFACT_ROOT, artifact["path"]))
                    == artifact["sha256"]
                    for artifact in generation_manifest.get("artifacts", {}).values()
                ),
            }
        )
    except Exception as error:
        verification["error"] = f"{type(error).__name__}: {error}"
    return verification


def pointer_matches_last_completed(results: list[dict[str, Any]]) -> bool:
    if not results:
        return False
    try:
        return results[-1]["pointer_path"].read_text(encoding="utf-8") == results[-1][
            "manifest_text"
        ]
    except Exception:
        return False


GENERATION_VERIFICATIONS = [verify_generation(result) for result in RUN_RESULTS]
COMPLETED_SEEDS = [result["seed"] for result in RUN_RESULTS]
POINTER_MATCHES_LAST_COMPLETED = pointer_matches_last_completed(RUN_RESULTS)

HGD_EXPORT_CHECKS: list[dict[str, Any]] = []


def final_check(name: str, condition: bool, observed: Any, expected: Any) -> None:
    HGD_EXPORT_CHECKS.append(
        {
            "check": name,
            "status": "PASS" if condition else "FAIL",
            "observed": observed,
            "expected": expected,
        }
    )


final_check("exact target seeds exported", COMPLETED_SEEDS == TARGET_SEEDS, COMPLETED_SEEDS, TARGET_SEEDS)
final_check(
    "one immutable generation per seed",
    len({result["bundle_id"] for result in RUN_RESULTS}) == len(TARGET_SEEDS),
    len({result["bundle_id"] for result in RUN_RESULTS}),
    len(TARGET_SEEDS),
)
final_check(
    "generation manifests verify",
    len(GENERATION_VERIFICATIONS) == len(TARGET_SEEDS)
    and all(
        row["seed_matches"] and row["run_config_hash_matches"] and row["bundle_matches"]
        for row in GENERATION_VERIFICATIONS
    ),
    GENERATION_VERIFICATIONS,
    "all target seed and bundle contracts match",
)
final_check(
    "all artifact hashes verify",
    len(GENERATION_VERIFICATIONS) == len(TARGET_SEEDS)
    and all(row["artifact_hashes_match"] for row in GENERATION_VERIFICATIONS),
    [row["seed"] for row in GENERATION_VERIFICATIONS if row["artifact_hashes_match"]],
    TARGET_SEEDS,
)
final_check(
    "pointer matches last completed generation",
    POINTER_MATCHES_LAST_COMPLETED,
    RUN_RESULTS[-1]["seed"] if POINTER_MATCHES_LAST_COMPLETED else None,
    COMPLETED_SEEDS[-1] if COMPLETED_SEEDS else None,
)
final_check("run registry complete", not RUN_FAILURES, RUN_FAILURES, [])

HGD_PASS = all(row["status"] == "PASS" for row in HGD_EXPORT_CHECKS)
display(pd.DataFrame(HGD_EXPORT_CHECKS))
display(Markdown("### Notebook 06 HGD model: " + ("READY" if HGD_PASS else "BLOCKED")))

if not HGD_PASS:
    raise RuntimeError("HGD multi-seed checks failed; inspect RUN_REGISTRY and HGD_EXPORT_CHECKS")

HGD_MANIFESTS = {result["seed"]: result["manifest"] for result in RUN_RESULTS}
HGD_MANIFEST = RUN_RESULTS[-1]["manifest"]
HGD_POINTER = RUN_RESULTS[-1]["pointer_path"]
EVALUATION_METRICS = RUN_RESULTS[-1]["evaluation_metrics"]
DIFFUSION_WEIGHTS = RUN_RESULTS[-1]["diffusion_weights"]
FUSION_GATES = RUN_RESULTS[-1]["fusion_gates"]
HGD_EVALUATION_METRICS = pd.concat(
    [
        result["evaluation_metrics"].assign(seed=result["seed"])
        for result in RUN_RESULTS
    ],
    ignore_index=True,
)

display(EVALUATION_METRICS[["phase", "rows", "positives", "threshold", "f1", "roc_auc"]])
display(DIFFUSION_WEIGHTS)
display(FUSION_GATES)
display(
    Markdown(
        "Every seed published a verified immutable bundle. The next cell compares these "
        "bundles against the notebook-04 EmerG bundles on the same protocol."
    )
)

In [ ]:
def emerg_manifest_paths() -> list[Path]:
    roots: list[Path] = []
    explicit = os.environ.get("COLDSTART_EMERG_ROOT")
    if explicit:
        roots.append(Path(explicit).expanduser())
    roots.extend([ARTIFACT_ROOT, PROJECT_ROOT / ".notebook" / "artifacts", INPUT_ROOT])
    found: dict[str, Path] = {}
    for root in roots:
        if not root.is_dir():
            continue
        for manifest_path in sorted(root.rglob("manifest.json")):
            parts = manifest_path.resolve().parts
            if len(parts) >= 4 and parts[-3] == "generations" and parts[-4] == "emerg-v1":
                found[str(manifest_path.resolve())] = manifest_path.resolve()
    return list(found.values())


def load_emerg_bundle(manifest_path: Path) -> dict[str, Any] | None:
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("model_status") != "PASS" or manifest.get("model_name") != "EmerG":
        return None
    if manifest.get("upstream_protocol", {}).get("pointer_sha256") != PROTOCOL_POINTER_SHA256:
        return None
    artifact = manifest["artifacts"]["evaluation_metrics"]
    metrics_path = manifest_path.parent / "evaluation_metrics.csv"
    if not metrics_path.is_file() or sha256_file(metrics_path) != artifact["sha256"]:
        return None
    contract_path = manifest_path.parent / "feature_contract.json"
    vocab_sha256 = None
    if contract_path.is_file():
        vocab_sha256 = json.loads(contract_path.read_text(encoding="utf-8")).get("vocab_sha256")
    return {
        "seed": int(manifest["run_config"]["seed"]),
        "bundle_id": manifest["bundle_id"],
        "created_at_utc": manifest["created_at_utc"],
        "metrics": pd.read_csv(metrics_path),
        "vocab_sha256": vocab_sha256,
    }


def paired_bootstrap_ci(deltas: np.ndarray, draws: int = 10_000, seed: int = 20260717) -> tuple[float, float]:
    if len(deltas) == 0:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    samples = rng.choice(deltas, size=(draws, len(deltas)), replace=True).mean(axis=1)
    return float(np.percentile(samples, 2.5)), float(np.percentile(samples, 97.5))


EMERG_BUNDLES: dict[int, dict[str, Any]] = {}
for path in emerg_manifest_paths():
    try:
        bundle = load_emerg_bundle(path)
    except Exception:
        bundle = None
    if bundle is None:
        continue
    incumbent = EMERG_BUNDLES.get(bundle["seed"])
    if incumbent is None or bundle["created_at_utc"] > incumbent["created_at_utc"]:
        EMERG_BUNDLES[bundle["seed"]] = bundle

COMPARISON_STATUS = "SKIPPED"
COMPARISON_NOTES: list[str] = []
PAIRED_SEEDS = sorted(set(EMERG_BUNDLES) & set(HGD_EVALUATION_METRICS["seed"]))

if not PAIRED_SEEDS:
    display(
        Markdown(
            "### EmerG comparison: SKIPPED\n"
            "No notebook-04 EmerG generation was found under the search roots that also "
            "carries this protocol pointer. Set `COLDSTART_EMERG_ROOT` to the artifact "
            "root that holds `models/ml-1m/emerg-v1/generations/`, or attach the "
            "notebook-04 output dataset, and re-run this cell."
        )
    )
else:
    emerg_metrics = pd.concat(
        [EMERG_BUNDLES[seed]["metrics"].assign(seed=seed) for seed in PAIRED_SEEDS],
        ignore_index=True,
    )
    columns = ["seed", "phase", "f1", "roc_auc"]
    paired = (
        HGD_EVALUATION_METRICS[columns]
        .merge(emerg_metrics[columns], on=["seed", "phase"], suffixes=("_hgd", "_emerg"))
        .assign(
            f1_delta=lambda frame: frame["f1_hgd"] - frame["f1_emerg"],
            auc_delta=lambda frame: frame["roc_auc_hgd"] - frame["roc_auc_emerg"],
        )
    )
    paired["phase"] = pd.Categorical(paired["phase"], categories=list(PHASES), ordered=True)

    by_phase = (
        paired.groupby("phase", observed=True)[["f1_hgd", "f1_emerg", "f1_delta", "auc_delta"]]
        .mean()
        .reset_index()
    )
    by_phase["hgd_wins"] = (
        paired.assign(win=paired["f1_delta"] > 0)
        .groupby("phase", observed=True)["win"]
        .sum()
        .to_numpy()
    )
    by_phase["seeds"] = (
        paired.groupby("phase", observed=True)["seed"].nunique().to_numpy()
    )

    by_seed = (
        paired.groupby("seed")[["f1_hgd", "f1_emerg", "f1_delta", "auc_delta"]].mean().reset_index()
    )
    seed_deltas = by_seed["f1_delta"].to_numpy(dtype=np.float64)
    ci_low, ci_high = paired_bootstrap_ci(seed_deltas)
    mean_delta = float(seed_deltas.mean())
    wins = int((seed_deltas > 0).sum())
    sign_test_p = float(0.5 ** len(seed_deltas)) if wins == len(seed_deltas) else float("nan")

    vocab_matches = [
        EMERG_BUNDLES[seed]["vocab_sha256"] == FEATURE_CONTRACT["vocab_sha256"]
        for seed in PAIRED_SEEDS
        if EMERG_BUNDLES[seed]["vocab_sha256"] is not None
    ]
    feature_parity = bool(vocab_matches) and all(vocab_matches)

    display(Markdown("#### Mean evaluation metrics per phase, paired over seeds"))
    display(by_phase)
    display(Markdown("#### Mean evaluation metrics per seed, averaged over phases"))
    display(by_seed)

    COMPARISON_STATUS = "HGD BEATS EMERG" if mean_delta > 0 and wins == len(seed_deltas) else (
        "HGD AHEAD ON AVERAGE" if mean_delta > 0 else "HGD DOES NOT BEAT EMERG"
    )
    if not feature_parity:
        COMPARISON_NOTES.append(
            "Feature vocabularies could not be confirmed identical to the EmerG bundle; "
            "the comparison may not isolate the method."
        )
    if len(seed_deltas) < len(TARGET_SEEDS):
        COMPARISON_NOTES.append(
            f"Only {len(seed_deltas)} of {len(TARGET_SEEDS)} target seeds are paired."
        )
    if ci_low <= 0.0 <= ci_high:
        COMPARISON_NOTES.append(
            "The paired bootstrap interval spans zero; with five seeds this is a weak "
            "interval and notebook 08 should be the arbiter."
        )

    display(
        Markdown(
            f"### EmerG comparison: {COMPARISON_STATUS}\n"
            f"- Mean F1 delta (HGD - EmerG), averaged over phases then seeds: **{mean_delta:+.4f}**\n"
            f"- Paired bootstrap 95% interval over {len(seed_deltas)} seeds: "
            f"[{ci_low:+.4f}, {ci_high:+.4f}]\n"
            f"- Seeds where HGD wins on mean F1: **{wins}/{len(seed_deltas)}**"
            + (f" (sign test p = {sign_test_p:.3f})" if np.isfinite(sign_test_p) else "")
            + "\n"
            f"- Identical feature vocabularies to EmerG: **{feature_parity}**\n"
            f"- Same protocol pointer as EmerG: **True** (enforced when loading bundles)"
            + ("\n\n" + "\n".join(f"- Caveat: {note}" for note in COMPARISON_NOTES) if COMPARISON_NOTES else "")
        )
    )

display(
    Markdown(
        "**Next notebook:** notebook 08 should read the `hgd-v1` bundles alongside "
        "`lightgcn-v1`, `emerg-v1` and `dgd-v1` and run the formal comparison there. "
        "This cell is a run-time sanity read, not the published result. For ablations, "
        "each ingredient is a single config edit: `id_dropout: 0.0` with a normal-init "
        "id embedding removes the cold-start prior, `fusion_logit_init: -40.0` removes "
        "the sparsemax branch, `modulation_gate_init: 0.0` removes DGD modulation, and "
        "`power_prior_bias: 40.0` pins every layer to the first-order graph."
    )
)
